## Notebook 2: Feature Engineering + Text Processing

This notebook focuses on preparing the participant profiles for the compatibility prediction task. We transform raw profile data into meaningful features, leveraging both textual and structured information to capture nuances in participant interactions.

### 1) Main Idea
Our primary goal is to create a rich set of features that can effectively represent the compatibility between two participants. This involves processing multi-value text fields using TF-IDF to capture semantic similarities in interests, objectives, and constraints, as well as generating pairwise features from structured data like age, industry, and company. The aim is to provide a comprehensive representation that a predictive model can use to assess compatibility.

### 2) Text Cleaning and Preprocessing
The three key text fields (`Business_Interests`, `Business_Objectives`, `Constraints`) are critical for understanding participant alignment. We first concatenate the `train` and `test` datasets to ensure consistent preprocessing across all profiles. Each text column undergoes a cleaning process:
*   Missing values are handled by replacing them with empty strings.
*   Text is converted to lowercase for standardization.
*   Semicolons, which act as delimiters for multiple values, are replaced with spaces.
*   Special characters are removed, and multiple spaces are condensed to single spaces.
*   The cleaned text is then stripped of leading/trailing whitespace.
This ensures that all textual data is in a clean, uniform format ready for vectorization.

### 3) TF-IDF Representation and Similarity Features
To convert the cleaned text into numerical features, we employ TF-IDF (Term Frequency-Inverse Document Frequency) vectorizers for each of the three text columns. Each vectorizer is fit on the entire corpus of cleaned text for its respective column, capturing the importance of words and phrases (up to bigrams).
*   `TfidfVectorizer` is configured with `max_features=5000` to limit dimensionality, `ngram_range=(1, 2)` to include single words and two-word phrases, and `stop_words='english'` to remove common, less informative words.
*   Once the TF-IDF vectors are generated for each participant, we can compute cosine similarity between the vectors of two participants (source and destination) for each text category. This cosine similarity serves as a powerful feature, quantifying how similar their interests, objectives, or constraints are.

### 4) Structured Pairwise Features
Beyond text, the structured fields (`Age`, `Gender`, `Role`, `Seniority`, `Company`, `Company Size`, `Industry`, `City`) also hold valuable information. We derive pairwise features by comparing the attributes of two participants:
*   **Numerical differences**: For `Age`, we calculate `age_diff` (absolute difference).
*   **Categorical matches**: For fields like `Company`, `Industry`, `City`, `Gender`, `Role`, and `Seniority`, we create binary features indicating whether the two participants share the same value (e.g., `same_company`, `same_industry`, `same_city`, `same_gender`, `same_role`, `same_seniority`).
These features provide direct, interpretable signals about structural commonalities or differences between participants.

### 5) Why this approach is practical (low compute, scalable)
This feature engineering approach is highly practical for several reasons:
*   **Low Compute**: TF-IDF vectorization is computationally efficient, especially with `max_features` limits. Generating pairwise features primarily involves straightforward comparisons and arithmetic operations.
*   **Scalable**: The methods used (TF-IDF, simple comparisons) scale well with an increasing number of participants. The text cleaning and feature generation can be parallelized or processed in batches if needed for very large datasets.
*   **Interpretability**: The resulting features, especially cosine similarity scores and binary match indicators, are relatively easy to interpret, offering insights into why certain participants might be deemed compatible.

### 6) Files Saved for later notebooks
To facilitate consistent feature generation in subsequent modeling notebooks and to avoid re-training the vectorizers, the trained TF-IDF models are serialized and saved:
*   `tfidf_interests.pkl`: The `TfidfVectorizer` fitted on `Business_Interests_clean`.
*   `tfidf_objectives.pkl`: The `TfidfVectorizer` fitted on `Business_Objectives_clean`.
*   `tfidf_constraints.pkl`: The `TfidfVectorizer` fitted on `Constraints_clean`.
These files allow for efficient loading and transformation of new data using the exact same vocabulary and weighting learned during this preprocessing step.

In [3]:
!pip -q install openpyxl joblib

import pandas as pd
import numpy as np
import re
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer



In [4]:
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = x.replace(";", " ")
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


In [5]:
train = pd.read_excel("train.xlsx")
test  = pd.read_excel("test.xlsx")
target = pd.read_csv("target.csv")

all_profiles = pd.concat([train, test], axis=0).reset_index(drop=True)

print("Train:", train.shape)
print("Test :", test.shape)
print("Target:", target.shape)
print("All profiles:", all_profiles.shape)


Train: (600, 12)
Test : (400, 12)
Target: (65203, 3)
All profiles: (1000, 12)


In [6]:
text_cols = ["Business_Interests", "Business_Objectives", "Constraints"]

for col in text_cols:
    all_profiles[col] = all_profiles[col].fillna("")
    all_profiles[col + "_clean"] = all_profiles[col].apply(clean_text)

print(all_profiles[[c + "_clean" for c in text_cols]].head())



                   Business_Interests_clean  \
0        product management venture capital   
1  startup ecosystem sales saas open source   
2                    startup ecosystem saas   
3                      ai startup ecosystem   
4                                      saas   

                           Business_Objectives_clean  \
0  looking for internship opportunities exploring...   
1  learning about industry trends looking for int...   
2  building professional visibility mentorship an...   
3                     learning about industry trends   
4                     networking with industry peers   

                                   Constraints_clean  
0            not evaluating service based businesses  
1                 prefer b2c or high growth startups  
2  looking for short term or project based engage...  
3  interested in growth or performance marketing ...  
4  prefer discussions with decision makers only l...  


In [7]:
tfidf_models = {}
tfidf_matrices = {}

for col in text_cols:
    corpus = all_profiles[col + "_clean"].tolist()

    vec = TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        stop_words="english",
        min_df=2
    )

    mat = vec.fit_transform(corpus)

    tfidf_models[col] = vec
    tfidf_matrices[col] = mat

print("TF-IDF models trained.")



TF-IDF models trained.


In [8]:
joblib.dump(tfidf_models["Business_Interests"], "tfidf_interests.pkl")
joblib.dump(tfidf_models["Business_Objectives"], "tfidf_objectives.pkl")
joblib.dump(tfidf_models["Constraints"], "tfidf_constraints.pkl")

print("Saved TF-IDF vectorizers:")
print("- tfidf_interests.pkl")
print("- tfidf_objectives.pkl")
print("- tfidf_constraints.pkl")


Saved TF-IDF vectorizers:
- tfidf_interests.pkl
- tfidf_objectives.pkl
- tfidf_constraints.pkl


In [9]:
print("Vocabulary sizes:")
print("Interests :", len(tfidf_models["Business_Interests"].vocabulary_))
print("Objectives:", len(tfidf_models["Business_Objectives"].vocabulary_))
print("Constraints:", len(tfidf_models["Constraints"].vocabulary_))


Vocabulary sizes:
Interests : 135
Objectives: 164
Constraints: 297
